In [9]:
#Markdown标题解析模块，并设计相应的实现方案：


# knowledge_engine.py - Markdown标题解析器模块

import re
import json
import yaml
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Tuple
from enum import Enum
from pathlib import Path
from datetime import datetime
import os


#Enum定义 卡片类别
class CardType(Enum):
    """卡片类型枚举"""
    MEMORY = "记忆"  # 记忆库卡片
    LEARNING = "学习"  # 学习研究
    PROFESSIONAL = "专业"  # 专业知识蒸馏
    THINKING = "思维"  # 思维蒸馏
    INTEREST = "兴趣"  # 兴趣蒸馏
  

  #  Enum定义 标题组件
class HeaderComponent(Enum):
    """标题组成部分枚举"""
    CATEGORY = "category"  # 分类（如：记忆）
    SUBJECT = "subject"  # 学科/主题（如：Python）
    SEQUENCE = "sequence"  # 序号（如：001）
    VERSION = "version"  # 版本号（可选）
  


In [5]:
#定义标准数据结构
@dataclass          #支撑了 yaml与json 作用于obsidian的.md
class ParsedHeader:
    """解析后的标题数据结构"""
    raw_header: str  # 原始标题字符串
    category: str  # 分类
    subject: str  # 主题
    sequence: str  # 序号
    version: Optional[str] = None  # 版本号（可选）
    card_type: Optional[CardType] = None  # 卡片类型
    components: Optional[Dict[str, str]] = None  # 所有组件字典
    metadata: Optional[Dict] = None  # 附加元数据                  据称包含提取、parsed_ar时间戳、original_code, 接入glm版本 parse_md_to_json & exttact_card_json
  
    def to_json(self, include_metadata: bool = True) -> Dict:
        """转换为JSON格式字典"""
        result = {
            "raw_header": self.raw_header,
            "category": self.category,
            "subject": self.subject,
            "sequence": self.sequence,
            "components": self.components or {},
            "card_type": self.card_type.value if self.card_type else None
        }
      
        if self.version:
            result["version"] = self.version
          
        if include_metadata and self.metadata:
            result["metadata"] = self.metadata
          
        return result
# 创建文件名与链接 
    def to_obsidian_link(self) -> str:
        """生成Obsidian双向链接格式"""
        return f"[[{self.raw_header}]]"
  
    def to_file_name(self, extension: str = "md") -> str:
        """生成标准文件名"""
        parts = [self.category, self.subject, self.sequence]
        if self.version:
            parts.append(f"v{self.version}")
        return f"{'-'.join(parts)}.{extension}"


In [18]:
#本cell中变量名为准则
class MarkdownHeaderParser:
    """Markdown标题解析器（支持【AAA-BBB-CCC】格式）"""
  
    def __init__(self, config_path: Optional[Path] = None):
        """
        初始化解析器
      
        Args:
            config_path: 配置文件路径，包含解析规则和映射关系
        """
        self._header_pattern = re.compile(
            r'###?\s*【([^】]+)】\s*(.+?)(?:\n|$)'
        )
        self._component_pattern = re.compile(
            r'([A-Za-z\u4e00-\u9fa5]+)-([A-Za-z\u4e00-\u9fa5]+)-(\d{3,})(?:-v(\d+))?'
        )
        self.config = self._load_config(config_path) if config_path else {}
      
    def _load_config(self, config_path: Path) -> Dict:
        """加载配置文件"""
        try:
            if config_path.suffix in ['.yaml', '.yml']:
                import yaml
                with open(config_path, 'r', encoding='utf-8') as f:
                    return yaml.safe_load(f)
            elif config_path.suffix == '.json':
                with open(config_path, 'r', encoding='utf-8') as f:
                    return json.load(f)
        except Exception as e:
            print(f"加载配置文件失败: {e}")
            return {}
 #定义标准数据结构
    def parse_header(self, markdown_content: str) -> List[ParsedHeader]:
        """
        从Markdown内容中提取所有符合格式的标题
      
        Args:
            markdown_content: Markdown文本内容
          
        Returns:
            List[ParsedHeader]: 解析后的标题对象列表
        """
        headers = []
        matches = self._header_pattern.findall(markdown_content)
      
        for match in matches:
            header_code = match[0].strip()  # 【AAA-BBB-CCC】部分
            title_text = match[1].strip()   # 标题文本部分
          
            parsed = self._parse_header_code(header_code, title_text)
            if parsed:
                headers.append(parsed)
              
        return headers
  
    def _parse_header_code(self, header_code: str, title_text: str) -> Optional[ParsedHeader]:
        """
        解析标题代码部分（【AAA-BBB-CCC】）
      
        Args:
            header_code: 标题代码字符串（如："Python-编程-001"）
            title_text: 标题文本部分
          
        Returns:
            ParsedHeader: 解析后的对象，如果格式无效则返回None
        """
        match = self._component_pattern.match(header_code)
        if not match:
            return None
          
        category, subject, sequence, version = match.groups()
      
        # 确定卡片类型
        card_type = self._determine_card_type(category, subject)
      
        # 构建组件字典
        components = {
            HeaderComponent.CATEGORY.value: category,
            HeaderComponent.SUBJECT.value: subject,
            HeaderComponent.SEQUENCE.value: sequence,
        }
      
        if version:
            components[HeaderComponent.VERSION.value] = version
      
        # 构建元数据
        metadata = {
            "parsed_at": datetime.now().isoformat(),
            "title_text": title_text,
            "original_code": header_code,
        }
      
        return ParsedHeader(
            raw_header=f"【{header_code}】{title_text}",
            category=category,
            subject=subject,
            sequence=sequence,
            version=version,
            card_type=card_type,
            components=components,
            metadata=metadata
        )

    #分类
    def _determine_card_type(self, category: str, subject: str) -> Optional[CardType]:
        """根据分类和主题确定卡片类型"""
        # 从配置文件中获取映射关系
        category_mapping = self.config.get("category_mapping", {})
      
        if category in category_mapping:
            type_str = category_mapping[category]
            try:
                return CardType(type_str)
            except ValueError:
                pass
              
        # 默认映射（如果没有配置文件）
        default_mapping = {
            "记忆": CardType.MEMORY,
            "学习": CardType.LEARNING,
            "专业": CardType.PROFESSIONAL,
            "思维": CardType.THINKING,
            "兴趣": CardType.INTEREST,
        }
      
        return default_mapping.get(category)

    #被建议修改，接入ai
    def generate_header(
        self,
        category: str,
        subject: str,
        sequence: str,
        title: str,
        version: Optional[str] = None
    ) -> str:
        """
        生成标准格式的Markdown标题
      
        Args:
            category: 分类
            subject: 主题
            sequence: 序号（3位数字）
            title: 标题文本
            version: 版本号（可选）
          
        Returns:
            str: 格式化的Markdown标题
        """
        # 验证序号格式
        if not re.match(r'^\d{3,}$', sequence):
            raise ValueError("序号必须是至少3位数字")
          
        header_code = f"{category}-{subject}-{sequence}"
        if version:
            if not re.match(r'^\d+$', version):
                raise ValueError("版本号必须是数字")
            header_code += f"-v{version}"
          
        return f"### 【{header_code}】{title}"
  
    def extract_and_convert(self, markdown_file: Path) -> Dict:
        """
        从Markdown文件中提取标题并转换为JSON结构
      
        Args:
            markdown_file: Markdown文件路径
          
        Returns:
            Dict: 包含文件信息和解析结果的JSON结构
        """
        try:
            with open(markdown_file, 'r', encoding='utf-8') as f:
                content = f.read()
          
            headers = self.parse_header(content)
          
            result = {
                "file_info": {
                    "path": str(markdown_file),
                    "name": markdown_file.name,
                    "size": markdown_file.stat().st_size,
                    "last_modified": datetime.fromtimestamp(
                        markdown_file.stat().st_mtime                     #实时调用
                    ).isoformat()
                },
                "headers": [header.to_json() for header in headers],
                "stats": {
                    "total_headers": len(headers),
                    "categories": {},
                    "subjects": {}
                }
            }
          
            # 统计信息
            for header in headers:
                # 按分类统计
                result["stats"]["categories"][header.category] = \
                    result["stats"]["categories"].get(header.category, 0) + 1
              
                # 按主题统计
                result["stats"]["subjects"][header.subject] = \
                    result["stats"]["subjects"].get(header.subject, 0) + 1
                  
            return result
        except Exception as e:
        # 👇 在这里增加判断：如果是 PermissionError，返回特定信息
            if isinstance(e, PermissionError):
                return {"error": "File is being written by Watcher", "file": str(markdown_file)}
        
        # 👇 如果不是 PermissionError，保持原来的错误处理逻辑不变
            return {
            "error": str(e),
            "file": str(markdown_file)
        }  
        
 #批处理 扫描watcher
    def batch_process_directory(self, directory: Path) -> Dict:
        """
        批量处理目录中的所有Markdown文件
      
        Args:
            directory: 目录路径
          
        Returns:
            Dict: 批量处理结果
        """
        results = {
            "processed_at": datetime.now().isoformat(),
            "directory": str(directory),
            "files": [],
            "summary": {
                "total_files": 0,
                "successful": 0,
                "failed": 0,
                "total_headers": 0
            }
        }
      
        if not directory.exists():
            return {**results, "error": "目录不存在"}
          
        md_files = list(directory.glob("**/*.md"))
        results["summary"]["total_files"] = len(md_files)
      
        for md_file in md_files:
            file_result = self.extract_and_convert(md_file)
          
            if "error" in file_result:
                results["summary"]["failed"] += 1
            else:
                results["summary"]["successful"] += 1
                results["summary"]["total_headers"] += len(file_result.get("headers", []))
          
            results["files"].append(file_result)
          
        return results

In [15]:
# 配置示例文件（parser_config.yaml）
def create_example_config():
    """创建示例配置文件"""
    config = {
        "category_mapping": {
            "记忆": "MEMORY",
            "学习": "LEARNING",
            "专业": "PROFESSIONAL",
            "思维": "THINKING",
            "兴趣": "INTEREST"
        },
        "subject_aliases": {
            "编程": ["开发", "编码", "计算机"],
            "Python": ["python", "PYTHON"],
            "JavaScript": ["JS", "javascript"]
        },
        "validation_rules": {                                  #灵活正则  取消硬编码映射
            "category_required": True,
            "subject_min_length": 2,
            "sequence_format": r'^\d{3,}$',
            "version_optional": True
        },
        "output_formats": {
            "json_indent": 2,
            "include_metadata": True,
            "timestamp_format": "%Y-%m-%d %H:%M:%S"
        }
    }
  
    config_path = Path("E:/Briah/config/parser_config.yaml")
    config_path.parent.mkdir(parents=True, exist_ok=True)
  
    import yaml
    with open(config_path, 'w', encoding='utf-8') as f:
        yaml.dump(config, f, allow_unicode=True, default_flow_style=False)
      
    print(f"示例配置文件已创建: {config_path}")

In [11]:
# # 使用示例函数
# def main_example():
#     """使用示例"""
  
#     # 1. 创建解析器（可选：传入配置文件）
#     config_path = Path("E:/Briah/config/parser_config.yaml")
#     parser = MarkdownHeaderParser(config_path)
  
#     # 2. 解析单个内容
#     example_content = """
#     # 我的知识库
  
#     ## 编程学习
  
#     ### 【Python-编程-001】装饰器详解
#     - **AI索引标签**：`装饰器` `Python`
#     - **归档时间**：2026-03-26
#     - **来源**：官方文档
  
#     装饰器用于扩展函数功能...
  
#     ### 【JavaScript-前端-002】ES6新特性
#     - **AI索引标签**：`JavaScript` `ES6`
#     这里是内容...
#     """
  
#     headers = parser.parse_header(example_content)
#     print(f"解析到 {len(headers)} 个标题")
  
#     # 转换为JSON
#     for header in headers:
#         json_data = header.to_json()
#         print(json.dumps(json_data, ensure_ascii=False, indent=2))
      
#     # 3. 处理单个文件
#     md_file = Path("E:/Briah/记忆库.md")
#     if md_file.exists():
#         result = parser.extract_and_convert(md_file)
      
#         # 保存解析结果
#         output_file = md_file.parent / f"{md_file.stem}_parsed.json"
#         with open(output_file, 'w', encoding='utf-8') as f:
#             json.dump(result, f, ensure_ascii=False, indent=2)
#         print(f"解析结果已保存到: {output_file}")
      
#     # 4. 批量处理目录
#     knowledge_dir = Path("E:/Briah")
#     batch_result = parser.batch_process_directory(knowledge_dir)
  
#     # 保存批量处理结果
#     batch_output = knowledge_dir / "知识库索引.json"
#     with open(batch_output, 'w', encoding='utf-8') as f:
#         json.dump(batch_result, f, ensure_ascii=False, indent=2)
  
#     # 5. 生成新标题
#     new_header = parser.generate_header(
#         category="Python",
#         subject="算法",
#         sequence="003",
#         title="快速排序算法详解",
#         version="1"
#     )
#     print(f"\n生成的标题: {new_header}")
    
    


In [14]:
create_example_config()
print(main_example())

示例配置文件已创建: E:\Briah\config\parser_config.yaml
解析到 2 个标题
{
  "raw_header": "【Python-编程-001】装饰器详解",
  "category": "Python",
  "subject": "编程",
  "sequence": "001",
  "components": {
    "category": "Python",
    "subject": "编程",
    "sequence": "001"
  },
  "card_type": null,
  "metadata": {
    "parsed_at": "2026-03-31T16:01:41.320175",
    "title_text": "装饰器详解",
    "original_code": "Python-编程-001"
  }
}
{
  "raw_header": "【JavaScript-前端-002】ES6新特性",
  "category": "JavaScript",
  "subject": "前端",
  "sequence": "002",
  "components": {
    "category": "JavaScript",
    "subject": "前端",
    "sequence": "002"
  },
  "card_type": null,
  "metadata": {
    "parsed_at": "2026-03-31T16:01:41.320175",
    "title_text": "ES6新特性",
    "original_code": "JavaScript-前端-002"
  }
}
解析结果已保存到: E:\Briah\记忆库_parsed.json

生成的标题: ### 【Python-算法-003-v1】快速排序算法详解
None


In [17]:
# if __name__ == "__main__":
#     # 创建示例配置
#     create_example_config()
  
#     # 运行示例
#     main_example()
# # ```

# # ## 功能特性说明：

# # ### 1. **智能标题解析**
# # - 支持标准格式：`### 【分类-主题-序号】标题文本`
# # - 可选版本号：`### 【分类-主题-序号-v版本】标题文本`
# # - 自动识别中英文分类和主题

# # ### 2. **结构化输出**
# # - 返回`ParsedHeader`数据类，包含完整组件
# # - 可转换为JSON、字典、Obsidian链接等多种格式
# # - 支持元数据附加

# # ### 3. **批量处理能力**
# # - 递归处理目录中所有Markdown文件
# # - 生成详细统计信息和索引
# # - 错误处理和容错机制

# # ### 4. **配置驱动**
# # - 通过YAML/JSON配置文件自定义映射规则
# # - 支持类别映射、别名处理、验证规则
# # - 灵活的输出格式配置

# # ### 5. **生成能力**
# # - 根据参数生成标准格式标题
# # - 自动验证序号和版本号格式
# # - 支持多种输出格式

# # ## 与知识库架构的集成：

# # 1. **与索引管理器集成**：
# # ```python
# # # index_manager.py中使用
# # from knowledge_engine import MarkdownHeaderParser      这个是外部引用才会有

# def build_index_with_parser():
#     parser = MarkdownHeaderParser()
#     index_data = parser.batch_process_directory(Path("E:/Briah"))
#     # 将结果集成到现有索引中

# # 2. **与监控系统集成**：

# # watcher.py中监听文件变化时使用
# def on_file_modified(file_path):
#     parser = MarkdownHeaderParser()
#     result = parser.extract_and_convert(file_path)
#     # 更新索引或通知AI


# # 3. **AI指令生成**：

# # AI生成标准化指令时使用
# def ai_generate_card_instruction():
#     parser = MarkdownHeaderParser()
#     header = parser.generate_header(
#         category="Python",
#         subject="装饰器",
#         sequence="001",
#         title="装饰器模式详解"
#     )
#     # 返回给write_card.py使用


# # 这个解析器模块可以作为您知识引擎的核心组件，为文件操作、索引构建、双向链接等功能提供结构化数据支持。

示例配置文件已创建: E:\Briah\config\parser_config.yaml
解析到 2 个标题
{
  "raw_header": "【Python-编程-001】装饰器详解",
  "category": "Python",
  "subject": "编程",
  "sequence": "001",
  "components": {
    "category": "Python",
    "subject": "编程",
    "sequence": "001"
  },
  "card_type": null,
  "metadata": {
    "parsed_at": "2026-03-31T16:05:13.170021",
    "title_text": "装饰器详解",
    "original_code": "Python-编程-001"
  }
}
{
  "raw_header": "【JavaScript-前端-002】ES6新特性",
  "category": "JavaScript",
  "subject": "前端",
  "sequence": "002",
  "components": {
    "category": "JavaScript",
    "subject": "前端",
    "sequence": "002"
  },
  "card_type": null,
  "metadata": {
    "parsed_at": "2026-03-31T16:05:13.170021",
    "title_text": "ES6新特性",
    "original_code": "JavaScript-前端-002"
  }
}
解析结果已保存到: E:\Briah\记忆库_parsed.json

生成的标题: ### 【Python-算法-003-v1】快速排序算法详解
